<a href="https://colab.research.google.com/github/Lahi070/CineSentiment-Mini-App/blob/main/CineSentiment_Data_Prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets

In [2]:
# Import the tool we just installed
from datasets import load_dataset

# Download the IMDb dataset
print("Downloading dataset...")
imdb_dataset = load_dataset("imdb")

print("Download complete!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Download complete!


In [3]:
import pandas as pd

# Convert the training data into a Pandas DataFrame (a visual table)
train_df = pd.DataFrame(imdb_dataset['train'])
test_df = pd.DataFrame(imdb_dataset['test'])

# Display the first 5 rows of the training data
print("Here are the first 5 movie reviews and their sentiment labels:")
train_df.head()

Here are the first 5 movie reviews and their sentiment labels:


,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [4]:
import re
import numpy as np

def clean_text(text):
    # Remove HTML break tags
    text = re.sub(r'<br\s*/?>', ' ', text)
    # Remove everything that isn't a letter or a space
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert all letters to lowercase
    return text.lower()

# Apply the cleaning function to all reviews in the training and testing sets
print("Cleaning training data...")
train_texts = [clean_text(text) for text in train_df['text']]
train_labels = np.array(train_df['label'])

print("Cleaning testing data...")
test_texts = [clean_text(text) for text in test_df['text']]
test_labels = np.array(test_df['label'])

print("Data cleaned! Here is a sample:")
print(train_texts[0][:100], "...") # Shows the first 100 characters of the first review


Cleaning training data...
Cleaning testing data...
Data cleaned! Here is a sample:
i rented i am curiousyellow from my video store because of all the controversy that surrounded it wh ...


In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# We will only learn the 10,000 most popular words to save memory
vocab_size = 10000
# Every review will be forced to be exactly 200 words long
max_length = 200

print("Tokenizing the text...")
# Create the tokenizer and fit it only on the training text
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

# Convert the text sentences into sequences of numbers
train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

print("Padding the sequences...")
# Pad the sequences with zeros at the end if they are too short, or cut them if too long
x_train = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
x_test = pad_sequences(test_sequences, maxlen=max_length, padding='post', truncating='post')

print("Phase 1 Complete!")
print(f"Shape of training data: {x_train.shape}")
print(f"Sample of numerical sequence:\n{x_train[0][:20]}")

Tokenizing the text...
Padding the sequences...
Phase 1 Complete!
Shape of training data: (25000, 200)
Sample of numerical sequence:
[  10 1540   10  237    1   35   55  390 1131   83    5   31    2 6941
   12 3274    9   51    9   13]


### Consolidated Data Preparation and Model Setup

In [6]:
# Import necessary libraries
from datasets import load_dataset
import pandas as pd
import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# --- 1. Load the IMDb dataset ---
print("Downloading and loading IMDb dataset...")
imdb_dataset = load_dataset("imdb")
print("Dataset loaded!")

# --- 2. Convert to Pandas DataFrames ---
print("Converting dataset to Pandas DataFrames...")
train_df = pd.DataFrame(imdb_dataset['train'])
test_df = pd.DataFrame(imdb_dataset['test'])
print("DataFrames created.")

# --- 3. Clean the text ---
def clean_text(text):
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.lower()

print("Cleaning text data...")
train_texts = [clean_text(text) for text in train_df['text']]
train_labels = np.array(train_df['label'])
test_texts = [clean_text(text) for text in test_df['text']]
test_labels = np.array(test_df['label'])
print("Text cleaning complete.")

# --- 4. Tokenize and Pad Sequences ---
vocab_size = 10000
max_length = 200

print("Tokenizing and padding sequences...")
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

x_train = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
x_test = pad_sequences(test_sequences, maxlen=max_length, padding='post', truncating='post')
print("Tokenization and padding complete. x_train and train_labels are now defined.")
print(f"Shape of training data (x_train): {x_train.shape}")
print(f"Shape of training labels (train_labels): {train_labels.shape}")

Dataset loaded!
Converting dataset to Pandas DataFrames...
DataFrames created.
Cleaning text data...
Text cleaning complete.
Tokenizing and padding sequences...
Tokenization and padding complete. x_train and train_labels are now defined.
Shape of training data (x_train): (25000, 200)
Shape of training labels (train_labels): (25000,)


In [7]:
# --- 5. Define the Keras LSTM Model ---
print("Defining Keras LSTM model...")
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_length),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])
print("Model architecture created!")
model.summary()

# --- 6. Compile the Model ---
print("Compiling the model...")
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
print("Model compiled and ready for training.")

Defining Keras LSTM model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model architecture created!


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Compiling the model...
Model compiled and ready for training.


### Regenerating Data Preparation Steps

To ensure `x_train` and `train_labels` are correctly defined in the kernel state, we will re-run the data preparation steps.

In [8]:
import pandas as pd
import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Assuming imdb_dataset is already loaded from previous steps
# Convert the training data into a Pandas DataFrame
train_df = pd.DataFrame(imdb_dataset['train'])
test_df = pd.DataFrame(imdb_dataset['test'])

# Define the text cleaning function
def clean_text(text):
    text = re.sub(r'<br\s*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.lower()

# Apply the cleaning function
train_texts = [clean_text(text) for text in train_df['text']]
train_labels = np.array(train_df['label'])
test_texts = [clean_text(text) for text in test_df['text']]
test_labels = np.array(test_df['label'])

print("DataFrames created and text cleaned.")
print(f"Shape of train_labels: {train_labels.shape}")

DataFrames created and text cleaned.
Shape of train_labels: (25000,)


In [9]:
# Define vocabulary size and max length (consistent with previous steps)
vocab_size = 10000
max_length = 200

print("Tokenizing the text...")
# Create the tokenizer and fit it only on the training text
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

# Convert the text sentences into sequences of numbers
train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

print("Padding the sequences...")
# Pad the sequences with zeros at the end if they are too short, or cut them if too long
x_train = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
x_test = pad_sequences(test_sequences, maxlen=max_length, padding='post', truncating='post')

print("Data preparation complete! x_train and train_labels are now defined.")
print(f"Shape of training data: {x_train.shape}")

Tokenizing the text...
Padding the sequences...
Data preparation complete! x_train and train_labels are now defined.
Shape of training data: (25000, 200)


In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Define the model
model = Sequential([
    # 1. Embedding Layer: Maps our 10,000 words into a 64-dimensional space
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_length),

    # 2. LSTM Layer: Reads the sequence of words (128 units of memory)
    LSTM(128, return_sequences=False),

    # 3. Dropout Layer: Randomly turns off neurons to prevent overfitting (memorizing the data)
    Dropout(0.5),

    # 4. Output Layer: Squeezes the result into a probability between 0 (Negative) and 1 (Positive)
    Dense(1, activation='sigmoid')
])

print("Model architecture created!")
model.summary()

Model architecture created!


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Compile the model
model.compile(
    loss='binary_crossentropy', # The standard loss function for binary (0 or 1) classification
    optimizer='adam',           # A highly efficient optimization algorithm
    metrics=['accuracy']        # We want to track the accuracy as it learns
)

print("Model compiled and ready for training.")

Model compiled and ready for training.


In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# We will only learn the 10,000 most popular words to save memory
vocab_size = 10000
# Every review will be forced to be exactly 200 words long
max_length = 200

# Define the model
model = Sequential([
    # 1. Embedding Layer: Maps our 10,000 words into a 64-dimensional space
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_length),

    # 2. LSTM Layer: Reads the sequence of words (128 units of memory)
    LSTM(128, return_sequences=False),

    # 3. Dropout Layer: Randomly turns off neurons to prevent overfitting (memorizing the data)
    Dropout(0.5),

    # 4. Output Layer: Squeezes the result into a probability between 0 (Negative) and 1 (Positive)
    Dense(1, activation='sigmoid')
])

print("Model architecture created!")
model.summary()

Model architecture created!


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
# Compile the model
model.compile(
    loss='binary_crossentropy', # The standard loss function for binary (0 or 1) classification
    optimizer='adam',           # A highly efficient optimization algorithm
    metrics=['accuracy']        # We want to track the accuracy as it learns
)

print("Model compiled and ready for training.")

Model compiled and ready for training.


In [14]:
# Train the model for 10 epochs (passes through the entire dataset)
# We use 20% of the training data as a validation set to check performance after each epoch
print("Starting training... This might take a few minutes!")

history = model.fit(
    x_train, train_labels,
    epochs=10,
    batch_size=64, # Train on 64 reviews at a time
    validation_split=0.2 # Hold back 20% of data for validation
)

print("Training complete!")

Starting training... This might take a few minutes!
Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.6227 - loss: 0.6612 - val_accuracy: 0.0054 - val_loss: 0.9008
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6367 - loss: 0.6387 - val_accuracy: 0.2770 - val_loss: 0.8527
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6968 - loss: 0.5751 - val_accuracy: 0.5194 - val_loss: 0.8804
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.7891 - loss: 0.4775 - val_accuracy: 0.8446 - val_loss: 0.5397
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.8529 - loss: 0.3759 - val_accuracy: 0.7054 - val_loss: 0.7537
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.8760 - loss: 0.3282 - val_accuracy: 0.7518 - val_loss: 0.6284
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9123 - loss: 0.2480 - val_accuracy: 0.7032 - val_loss: 0.8236
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 1

In [15]:
import pickle
from google.colab import files

# 1. Save the model
model.save('cinesentiment_model.h5')

# 2. Save the tokenizer
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

# 3. Download to your computer
files.download('cinesentiment_model.h5')
files.download('tokenizer.pickle')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
import gradio as gr
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re

# Clean the text
def clean_text(text):
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.lower()

# Predict using the model you already trained above
def predict_sentiment(review):
    cleaned = clean_text(review)
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(sequence, maxlen=200, padding='post', truncating='post')

    prediction = model.predict(padded)[0][0]
    return {"Positive": float(prediction), "Negative": float(1.0 - prediction)}

# Build the interface
app = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=5, placeholder="Type a movie review here..."),
    outputs=gr.Label(num_top_classes=2),
    title="CineSentiment: Movie Review Classifier"
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://94bb8487e47f6ff7cc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Evaluate Model Performance

After training, it's crucial to evaluate the model's performance on a separate test set to understand how well it generalizes to new, unseen data.

In [17]:
print("Evaluating model performance on test data...")
loss, accuracy = model.evaluate(x_test, test_labels, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print("Model evaluation complete.")

Evaluating model performance on test data...
Test Loss: 0.6731
Test Accuracy: 0.7948
Model evaluation complete.
